<a href="https://colab.research.google.com/github/uwol1116/GenAI-Class/blob/main/Assignment3_1_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 1. Transformer (Text Modeling)

In [13]:
!pip install datasets -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from collections import Counter
from datasets import load_dataset

dataset = load_dataset("imdb")
dataset = dataset["train"].select(range(1000))
texts = dataset["text"]


In [14]:
def build_vocab(texts, max_vocab=10000):
    counter = Counter()
    for text in texts:
        counter.update(text.lower().split())
    vocab = {"<pad>": 0, "<unk>": 1, "<sos>": 2, "<eos>": 3}
    for word, _ in counter.most_common(max_vocab - 4):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(texts)
vocab_size = len(vocab)
idx2word = {v: k for k, v in vocab.items()}

def tokenize(text, vocab, max_len=64):
    tokens = text.lower().split()[:max_len]
    return [vocab.get(t, 1) for t in tokens]

print(f"Vocab size: {vocab_size}")


Vocab size: 10000


## 1-1. Positional Encoding

In [15]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

D_MODEL = 64
pe_module = PositionalEncoding(d_model=D_MODEL)
dummy = torch.zeros(1, 20, D_MODEL)
pe_out = pe_module(dummy)[0]  # (20, 64)

print(f"PE shape: {pe_out.shape}")
print(f"Position  0 (first 8 dims): {pe_out[0,  :8].detach().numpy().round(4)}")
print(f"Position 10 (first 8 dims): {pe_out[10, :8].detach().numpy().round(4)}")


PE shape: torch.Size([20, 64])
Position  0 (first 8 dims): [0. 1. 0. 1. 0. 1. 0. 1.]
Position 10 (first 8 dims): [-0.544  -0.8391  0.9376  0.3476 -0.6129  0.7901 -0.8798 -0.4754]


0을 대입했을 때 sin(0)=0, cos(0)=1이기에 짝수 차원은 0에 가깝고 홀수 차원은 1에 가까운 값이 나온다.

10을 대입했을 때 각 차원의 주기가 서로 달라서 어떤 차원은 sin 값이 크게 변화하고 어떤 차원은 cos 값이 작게 변하는 식으로 변화한다.

낮은 차원(i=0)에서는 주기가 짧으므로 값이 빠르게 바뀌고 높은 차원일수록 긴 주기를 가지고 있어 천천히 변화한다.

각각의 차원이 다른 주기를 가짐으로써 모델은 절대적 위치와 상대적 위치 정보를 다 학습할 수 있다.

## 1-2. Masking Experiment

In [16]:
def create_causal_mask(seq_len):
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
    return mask.masked_fill(mask == 1, float('-inf'))

class DecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_ff=128, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(nn.Linear(d_model, dim_ff), nn.ReLU(), nn.Linear(dim_ff, d_model))
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, use_mask=True):
        mask = create_causal_mask(x.size(1)).to(x.device) if use_mask else None
        attn_out, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.drop(self.ff(x)))
        return x

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.emb    = nn.Embedding(vocab_size, d_model)
        self.pe     = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, nhead) for _ in range(num_layers)])
        self.head   = nn.Linear(d_model, vocab_size)

    def forward(self, x, use_mask=True):
        x = self.pe(self.emb(x))
        for layer in self.layers:
            x = layer(x, use_mask)
        return self.head(x)

def generate_sequence(model, prompt_tokens, max_len=20, use_mask=True):
    model.eval()
    generated = prompt_tokens.copy()
    with torch.no_grad():
        for _ in range(max_len):
            inp    = torch.tensor([generated])
            logits = model(inp, use_mask=use_mask)   # (1, T, vocab_size)
            next_token = logits[0, -1].argmax().item()  # 마지막 위치의 예측
            generated.append(next_token)
            if next_token == vocab["<eos>"]:
                break
    return generated

decoder = TransformerDecoder(vocab_size, d_model=64, nhead=4, num_layers=2)

prompt = tokenize(texts[0], vocab, max_len=5)  # 앞 5토큰을 프롬프트로 사용

seq_masked   = generate_sequence(decoder, prompt, max_len=15, use_mask=True)
seq_unmasked = generate_sequence(decoder, prompt, max_len=15, use_mask=False)

print(f"Prompt tokens   : {prompt}")
print(f"Prompt words    : {[idx2word.get(t, '<unk>') for t in prompt]}")
print()
print(f"With mask    generated : {seq_masked}")
print(f"With mask    words     : {[idx2word.get(t, '<unk>') for t in seq_masked]}")
print()
print(f"Without mask generated : {seq_unmasked}")
print(f"Without mask words     : {[idx2word.get(t, '<unk>') for t in seq_unmasked]}")

Prompt tokens   : [12, 935, 12, 208, 5287]
Prompt words    : ['i', 'rented', 'i', 'am', 'curious-yellow']

With mask    generated : [12, 935, 12, 208, 5287, 6410, 7301, 3243, 6413, 7445, 7404, 3133, 601, 6211, 4489, 9617, 4771, 6211, 6204, 9547]
With mask    words     : ['i', 'rented', 'i', 'am', 'curious-yellow', '/>and,', 'format', 'disgusted', 'dumb,', 'frontier', 'unimpressive', 'screen.<br', 'single', 'candidate', 'kidnap', 'picture,', 'laws', 'candidate', 'marginally', 'anywhere,']

Without mask generated : [12, 935, 12, 208, 5287, 9709, 5032, 8149, 1776, 1753, 4840, 5550, 8570, 2917, 8567, 9082, 7301, 7936, 67, 3249]
Without mask words     : ['i', 'rented', 'i', 'am', 'curious-yellow', 'suddenly,', 'lady,', 'incest', 'field', 'soul', 'celie', 'irrational', 'february', 'about,', "/>here's", 'squeeze', 'format', 'project,', 'can', 'strongly']


Position​‍​‌‍​‍‌ 0인 경우 sin(0)=0, cos(0)=1이기 때문에 짝수 차원은 0, 홀수 차원은 1에 가까운 값을 가진다.

Position 10인 경우 각 차원의 주파수가 서로 다르기 때문에 sin과 cos 값이 차원마다 다양하게 변화한다.

낮은 차원(i=0)일수록 주파수가 높아 위치에 따라 빠르게 변화하고, 높은 차원일수록 주파수가 낮아 천천히 변화한다.

마스킹을​‍​‌‍​‍‌ 적용한 경우 각 위치에서 현재 토큰과 이전 토큰만 참조할 수 있기 때문에 auto-regressive 시퀀스 생성에 적합한 ​‍​‌‍​‍‌구조이다.

마스킹을 적용하지 않은 경우 미래의 토큰까지 참조할 수 있기 때문에 information leakage가 발생하며, 실제 추론 시 미래 정보를 알 수 없으므로 훈련과 추론 사이의 불일치가 발생한다.

두 결과가 다른 이유는 마스킹 유무에 따라 각 토큰이 참조하는 범위가 달라지고, 그 결과로 생성되는 representation이 달라지기 때문에 최종 예측 시퀀스도 다르게 나타난다.

이처럼 차원마다 다른 주파수를 사용하기 때문에 모델은 절대적 위치와 상대적 위치 정보를 모두 학습할 수 ​‍​‌‍​‍‌있다.

## 1-3. Attention Analysis

In [17]:
class DecoderLayerWithAttn(nn.Module):
    # attention score를 직접 계산하여 softmax 전/후 값을 반환하는 Decoder Layer
    def __init__(self, d_model, nhead, dim_ff=128):
        super().__init__()
        self.nhead = nhead
        self.d_k   = d_model // nhead
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, dim_ff), nn.ReLU(), nn.Linear(dim_ff, d_model))

    def forward(self, x, use_mask=True):
        B, T, D = x.shape
        H, dk   = self.nhead, self.d_k

        Q = self.Wq(x).view(B, T, H, dk).transpose(1, 2)  # (B, H, T, dk)
        K = self.Wk(x).view(B, T, H, dk).transpose(1, 2)
        V = self.Wv(x).view(B, T, H, dk).transpose(1, 2)

        scores_before = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(dk)  # softmax 전

        if use_mask:
            mask = create_causal_mask(T).to(x.device)
            scores_before = scores_before + mask.unsqueeze(0).unsqueeze(0)

        scores_after = F.softmax(scores_before, dim=-1)                        # softmax 후

        out = torch.matmul(scores_after, V)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        out = self.Wo(out)

        x = self.norm1(x + out)
        x = self.norm2(x + self.ff(x))
        return x, scores_before, scores_after

class TransformerDecoderWithAttn(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.emb    = nn.Embedding(vocab_size, d_model)
        self.pe     = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderLayerWithAttn(d_model, nhead) for _ in range(num_layers)])
        self.head   = nn.Linear(d_model, vocab_size)

    def forward(self, x, use_mask=True):
        x = self.pe(self.emb(x))
        all_scores_before, all_scores_after = [], []
        for layer in self.layers:
            x, sb, sa = layer(x, use_mask)
            all_scores_before.append(sb)
            all_scores_after.append(sa)
        return self.head(x), all_scores_before, all_scores_after

attn_decoder = TransformerDecoderWithAttn(vocab_size, d_model=64, nhead=4, num_layers=2)
attn_decoder.eval()

sample_input = torch.tensor([tokenize(texts[0], vocab, max_len=10)])

with torch.no_grad():
    _, scores_before_list, scores_after_list = attn_decoder(sample_input, use_mask=True)

# Layer 0, Head 0 기준 출력
sb = scores_before_list[0]  # (1, 4, 10, 10)
sa = scores_after_list[0]

print(f"Attention score shape: {sb.shape}  (batch, heads, seq, seq)")
print()
print("[Layer 0, Head 0] Before softmax (5x5):")
print(sb[0, 0, :5, :5].numpy().round(3))
print()
print("[Layer 0, Head 0] After softmax (5x5):")
print(sa[0, 0, :5, :5].numpy().round(3))
print()

# 각 토큰이 받는 평균 attention 확인 (after softmax, head 평균)
avg_attn = sa[0].mean(dim=0)  # (10, 10) - head 평균
token_importance = avg_attn.mean(dim=0)  # 각 토큰이 받는 평균 attention
tokens_words = [idx2word.get(t, '<unk>') for t in sample_input[0].tolist()]
print("Token importance (avg attention received):")
for word, score in zip(tokens_words, token_importance.tolist()):
    print(f"  {word:20s} : {score:.4f}")

Attention score shape: torch.Size([1, 4, 10, 10])  (batch, heads, seq, seq)

[Layer 0, Head 0] Before softmax (5x5):
[[-0.504   -inf   -inf   -inf   -inf]
 [-0.023 -0.606   -inf   -inf   -inf]
 [-0.621 -0.687 -0.366   -inf   -inf]
 [ 0.238 -0.165  0.122  0.17    -inf]
 [-0.532 -0.166 -0.653 -0.445  0.138]]

[Layer 0, Head 0] After softmax (5x5):
[[1.    0.    0.    0.    0.   ]
 [0.642 0.358 0.    0.    0.   ]
 [0.31  0.29  0.4   0.    0.   ]
 [0.286 0.191 0.255 0.268 0.   ]
 [0.157 0.226 0.139 0.171 0.307]]

Token importance (avg attention received):
  i                    : 0.2881
  rented               : 0.1731
  i                    : 0.1385
  am                   : 0.1075
  curious-yellow       : 0.0888
  from                 : 0.0829
  my                   : 0.0572
  video                : 0.0366
  store                : 0.0186
  because              : 0.0087


softmax 이전의 score는 Q와 K의 내적을 sqrt(d_k)로 나눈 실수 값이며 두 토큰 간의 유사도를 나타낸다.

softmax 이후의 score는 각 행의 합이 1이 되는 확률 분포로 변환되며 해당 위치에서 각 토큰에 얼마나 집중하는지를 나타낸다.

특정 토큰이 높은 attention을 받는 이유는 해당 토큰의 Key 벡터가 Query 벡터와 내적했을 때 큰 값을 가지기 때문이다. 이는 의미적으로 관련성이 높은 단어일수록 Q-K 공간에서 유사한 방향을 가지기 때문에 발생한다.

score를 sqrt(d_k)로 나누는 이유는 차원 수가 커질수록 내적값의 크기가 커져 softmax의 기울기가 매우 작아지는 gradient 소멸 문제를 방지하기 위함이다.

## 1-4. Model Modification (Number of Heads)

In [18]:
print("nhead 변경 실험 (d_model=64 고정)")
print("-" * 50)

for nhead in [1, 4, 8]:
    try:
        m = TransformerDecoder(vocab_size, d_model=64, nhead=nhead, num_layers=2)
        prompt = tokenize(texts[0], vocab, max_len=5)
        seq = generate_sequence(m, prompt, max_len=15, use_mask=True)
        words = [idx2word.get(i, '<unk>') for i in seq]
        print(f"nhead={nhead}: {words}")
    except Exception as e:
        print(f"nhead={nhead}: Error - {e}")

nhead 변경 실험 (d_model=64 고정)
--------------------------------------------------
nhead=1: ['i', 'rented', 'i', 'am', 'curious-yellow', 'meerkats', 'expected', 'forgot', 'vampires', 'photographs', "should've", 'nowhere', 'unrealistic', 'area', 'touched', 'ground.', 'additionally,', 'death,', 'competition.', 'near']
nhead=4: ['i', 'rented', 'i', 'am', 'curious-yellow', 'begging', 'frankly', 'distraction', 'criminals', 'episode.', 'standard,', 'pathetic.<br', 'stan', 'right', 'arrogant', 'entire', 'girlfriend', 'say,', 'right.', 'life"']
nhead=8: ['i', 'rented', 'i', 'am', 'curious-yellow', 'deathly', 'again?', 'fanciful', 'smarter', 'suspense', 'opera.', 'hang', 'there', 'simplest', 'scratch', 'repeat', 'disappointingly', 'slapstick', 'guitar', 'elisabeth']


nhead=1인 경우 하나의 attention head만 존재하기 때문에 토큰 간의 관계를 단일한 관점에서만 파악할 수 있으며, 표현력이 제한된다.

nhead=4인 경우 구문적 관계, 의미적 관계, 위치 관계 등 서로 다른 4가지 관점을 병렬로 학습할 수 있기 때문에 균형 잡힌 표현이 가능하다.

nhead=8인 경우 더 세밀한 관계를 포착할 수 있지만, d_model=64를 8개로 나누면 각 head의 차원(d_k=8)이 너무 작아지기 때문에 개별 head가 담을 수 있는 정보량이 줄어드는 단점이 있다.

세 결과가 서로 다른 이유는 head 수에 따라 attention 패턴이 달라지기 때문이다. 각 head의 차원과 개수가 달라지면 모델이 학습하는 토큰 간 관계의 형태가 달라지고, 그 결과 생성되는 representation과 최종 시퀀스도 다르게 나타난다.
